In [1]:
# 데이터 로더 구축
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FastEmotionDataset(Dataset):
    def __init__(self, cropped_base, split='train', transform=None):
        self.transform = transform
        self.samples = []
        self.label_map = {'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3, 'neutral': 4}
        
        split_dir = os.path.join(cropped_base, split)
        
        for emotion_folder in os.listdir(split_dir):
            if emotion_folder not in self.label_map: continue
            
            folder_path = os.path.join(split_dir, emotion_folder)
            for fname in os.listdir(folder_path):
                self.samples.append({
                    'img_path': os.path.join(folder_path, fname),
                    'label': self.label_map[emotion_folder]
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        image = cv2.imread(item['img_path'])
        if image is None:
            return self.__getitem__(0) # 손상 파일 무시 로직
            
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
            
        return image, torch.tensor(item['label'], dtype=torch.long)

# 데이터 증강 (Augmentation)
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

CROPPED_BASE = "/workspace/user1/dataset_v3_cropped"
train_dataset = FastEmotionDataset(CROPPED_BASE, split='train', transform=train_transform)
val_dataset = FastEmotionDataset(CROPPED_BASE, split='val', transform=val_transform)

# A40 절반 자원 사용 설정
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

print(f"⚡ 데이터 로더 세팅 완료! Train: {len(train_dataset)} / Val: {len(val_dataset)}")

⚡ 데이터 로더 세팅 완료! Train: 5882 / Val: 1163


In [2]:
# 3대 모델 벤치마크 및 학습 루프 (EfficientNet, ResNet+DAN, ViT)

import torch.nn as nn
from torchvision import models
import timm
from tqdm import tqdm
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =====================================================================
# 1. 아키텍처 정의
# =====================================================================

# A. EfficientNet-B2
class EfficientNetB2_FER(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.model = timm.create_model('efficientnet_b2', pretrained=True, num_classes=num_classes)
    def forward(self, x):
        return self.model(x)

# B. ResNet50 + Spatial Attention (DAN-Lite)
class SpatialAttention(nn.Module):
    def __init__(self, in_planes):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.cat([avg_out, max_out], dim=1)
        attn = self.conv1(attn)
        return x * self.sigmoid(attn)

class ResNet50_DAN_Lite(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        backbone = models.resnet50(weights='IMAGENET1K_V2')
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.attention = SpatialAttention(2048)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(2048, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.attention(x)
        x = self.pool(x)
        return self.classifier(x)

# C. Vision Transformer (ViT-Base)
class ViT_FER(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes, drop_rate=0.3)
    def forward(self, x):
        return self.model(x)

# =====================================================================
# 2. 통합 학습 및 평가 함수
# =====================================================================
def train_and_evaluate(model_name, model, train_loader, val_loader, device):
    print(f"\n{'='*50}\n🚀 모델 학습 시작: {model_name}\n{'='*50}")
    
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # ViT는 구조적 특성상 학습률을 낮춰야 안정적으로 학습됨
    lr = 1e-4 if 'ViT' in model_name else 3e-4
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scaler = torch.amp.GradScaler('cuda')

    num_epochs = 30
    patience = 5
    best_val_acc = 0.0
    best_val_loss = float('inf')
    trigger_times = 0
    
    save_dir = '/workspace/user1/model'
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f'best_{model_name}.pth')

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in tqdm(train_loader, desc=f"{model_name} - Ep [{epoch+1}/{num_epochs}] Train", leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)

        # 검증 루프
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc = correct / total * 100
        
        print(f"[{model_name}] Ep {epoch+1} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        # 조기 종료 및 모델 저장
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            trigger_times = 0
            torch.save(model.state_dict(), save_path)
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f"🛑 {model_name} 조기 종료 (Best Acc: {best_val_acc:.2f}%)")
                break
                
    return best_val_acc

# =====================================================================
# 3. 벤치마크 실행부
# =====================================================================
models_to_test = {
    "EfficientNet-B2": EfficientNetB2_FER(num_classes=5),
    "ResNet50_DAN_Lite": ResNet50_DAN_Lite(num_classes=5),
    "ViT_Base": ViT_FER(num_classes=5)
}

results = {}

for name, model_arch in models_to_test.items():
    best_acc = train_and_evaluate(name, model_arch, train_loader, val_loader, device)
    results[name] = best_acc
    
    # 다음 모델 학습을 위해 OOM 방지용 GPU 메모리 비우기
    del model_arch
    torch.cuda.empty_cache()

# 최종 결과 비교 출력
print("\n" + "*"*40)
print("🏆 3대 모델 벤치마크 최종 결과")
print("*"*40)
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Best Val Accuracy (%)'])
results_df = results_df.sort_values(by='Best Val Accuracy (%)', ascending=False).reset_index(drop=True)
print(results_df)

model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/user1/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 517MB/s]


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


🚀 모델 학습 시작: EfficientNet-B2


[EfficientNet-B2] Ep 1 | Val Loss: 1.4071 | Val Acc: 58.04%


... (47 outputs left out)

[ViT_Base] Ep 10 | Val Loss: 1.0779 | Val Acc: 68.79%


[ViT_Base] Ep 11 | Val Loss: 1.0835 | Val Acc: 66.98%


[ViT_Base] Ep 12 | Val Loss: 1.0767 | Val Acc: 68.44%


[ViT_Base] Ep 13 | Val Loss: 1.1241 | Val Acc: 68.27%
🛑 ViT_Base 조기 종료 (Best Acc: 69.48%)

****************************************
🏆 3대 모델 벤치마크 최종 결과
****************************************
               Model  Best Val Accuracy (%)
0  ResNet50_DAN_Lite              71.281169
1    EfficientNet-B2              69.905417
2           ViT_Base              69.475494
